In [1]:
! pip install segmentation_models_pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 5.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.3 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found existing 

In [2]:
# All imports
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import pickle
from os.path import splitext
from os import listdir
import numpy as np
from glob import glob
import torch
from torch.utils.data import Dataset
import logging
from PIL import Image
import tifffile as tiff


import torch
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch import Tensor
import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/usr/local/lib/python3.11/dist-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.6' (you have '2.0.4'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [3]:
class BasicDataset(Dataset):
    def __init__(self, imgs_dir, scale=1):
        self.imgs_dir = imgs_dir
        self.scale = scale
        assert 0 < scale <= 1, 'Scale must be between 0 and 1'
        
        self.ids = [splitext(file)[0] for file in listdir(imgs_dir)
                    if not file.startswith('.')]
        self.ids.sort()        
        print(len(self.ids))

    def __len__(self):
        return len(self.ids)

    def preprocess(cls, tif_img, scale):
        img_nd = np.array(tif_img, dtype=np.float32)
        # Ensure 3D (H,W,channels)
        if img_nd.ndim == 2:
            img_nd = np.expand_dims(img_nd, axis=2)
        # Channel-first conversion: (C, H, W)
        img_trans = img_nd.transpose((2, 0, 1))

        # Subtract per-band minimum so each channel starts from zero
        # shape: (C, 1, 1)
        _, height, width = img_trans.shape
        if height == 384 and width == 384:
            img_trans = np.clip((img_trans - 5000) / 3, a_min=0, a_max=None)
        band_max = img_trans.max(axis=(1, 2), keepdims=True)
        img_trans = img_trans / np.maximum(band_max, 1)

        return img_trans

    def __getitem__(self, i):
        idx = self.ids[i]
        img_file = glob(f"{self.imgs_dir}/{idx}.tif")

        assert len(img_file) == 1, \
            f'Either no image or multiple images found for the ID {idx}: {img_file}'
        img = tiff.imread(img_file[0])
        img = self.preprocess(img, self.scale)
        return {
            'image': torch.from_numpy(img).type(torch.FloatTensor),
        }


In [4]:
def create_model(in_channels=4, device=device):
    model = smp.DeepLabV3Plus(
        encoder_name='resnet50',
        encoder_weights='imagenet',
        in_channels=3,
        classes=1,
        activation='sigmoid'
    )
    original_conv = model.encoder.conv1
    new_conv = nn.Conv2d(
        in_channels,
        original_conv.out_channels,
        kernel_size=original_conv.kernel_size,
        stride=original_conv.stride,
        padding=original_conv.padding,
        bias=False
    )
    with torch.no_grad():
        new_conv.weight[:, :3, :, :] = original_conv.weight.clone()
        if in_channels > 3:
            new_conv.weight[:, 3:, :, :] = original_conv.weight.mean(dim=1, keepdim=True).clone().repeat(1, in_channels - 3, 1, 1)
    model.encoder.conv1 = new_conv
    model.to(device)
    print(f"Model created with {in_channels} input channels and moved to {device}.")
    return model

In [5]:
dir_img = "/kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data"
dataset = BasicDataset(dir_img)


316


In [6]:
model_path = "/kaggle/input/finalmodel/pytorch/default/1/final_model.pth"
model = create_model(in_channels=4, device=device)
model.load_state_dict(torch.load(model_path, map_location=device))
print(f"Successfully loaded weights from {model_path}")
model.eval()

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Model created with 4 input channels and moved to cuda.


/tmp/ipykernel_31/4207824518.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


Successfully loaded weights from /kaggle/input/finalmodel/pytorch/default/1/final_model.pth


DeepLabV3Plus(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(4, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequentia

In [7]:
def predict_img(net,
                full_img,
                device,
                scale_factor=1,
                out_threshold=0.5):
    net.eval()
    img = full_img
    img = img.unsqueeze(0)
    img = img.to(device=device)
    with torch.no_grad():
        mask = net(img)
    with torch.no_grad():
        mask = net(img)> out_threshold
    return mask.cpu().squeeze().numpy()

In [8]:
import numpy as np

def rle_encode(mask):
    """
    Encodes a binary mask using Run-Length Encoding (RLE).    
    Args:
        mask (np.ndarray): 2D binary mask (0s and 1s).
    Returns:
        str: RLE-encoded string, or a single space " " if mask is all zeros.
    """
    if np.sum(mask) == 0:
        return " "  # As it seems that kaggle reject nulls. We'll handle cloud-free images with empty spaces.
    
    pixels = mask.flatten(order='F')  # Flatten in column-major order
    pixels = np.concatenate([[0], pixels, [0]])  # Add padding to detect transitions
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1  # Get transition indices
    runs[1::2] -= runs[::2]  # Compute run lengths
    runs[::2] -= 1  # Make it 0-indexed instead of 1-indexed

    return " ".join(map(str, runs))  # Convert to string format

def rle_decode(mask_rle: str, shape=(256, 256)) -> np.ndarray:
    """Decodes an RLE-encoded string into a binary mask with validation checks."""
    
    if not isinstance(mask_rle, str) or not mask_rle.strip() or mask_rle.lower() == 'nan':
        # Return all-zero mask if RLE is empty, invalid, or NaN
        return np.zeros(shape, dtype=np.uint8)
    
    try:
        s = list(map(int, mask_rle.split()))
    except:
        raise Exception("RLE segmentation must be a string and containing only integers")
    
    if len(s) % 2 != 0:
        raise Exception("RLE segmentation must have even-length (start, length) pairs")
    
    if any(x < 0 for x in s):
        raise Exception("RLE segmentation must not contain negative values")
    
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    starts, lengths = s[0::2], s[1::2]
    
    for start, length in zip(starts, lengths):
        if start >= mask.size or start + length > mask.size:
            raise Exception("RLE indices exceed image size")
        mask[start:start + length] = 1
    
    return mask.reshape(shape, order='F')  # Convert to column-major order

In [38]:
from skimage.transform import resize

total_score = 0.0
records = []
i=0;
# last_x=100
for x in range(0,len(dataset)):
    # if(i>50): break
    i+=1
    img=dataset[x]['image']
    out=predict_img(model,img,device,out_threshold=0.5)
    out = resize(out, (256, 256), order=0, preserve_range=True, anti_aliasing=False).astype(np.uint8)
    img_np = img.cpu().numpy().transpose(1, 2, 0) 
    band_names = ['R','G','B','IR']
    last_x=x
    print(dataset.ids[x])
    # Show images
    # fig, axes = plt.subplots(1, 2, figsize=(24, 6))
    # # Build and normalize the RGB composite
    # rgb = np.stack([
    #     img_np[..., 0],
    #     img_np[..., 1], 
    #     img_np[..., 2], 
    # ], axis=-1).astype(float)
    # rgb /= rgb.max()
    # print(dataset.ids[x])
    # axes[0].imshow(rgb)
    # axes[0].set_title('RGB composite')
    # axes[0].axis('off')
    
    # # Show the predicted mask
    # axes[1].imshow(out,  vmin=0, vmax=1)
    # axes[1].set_title("Predicted Mask")
    # axes[1].axis('off')
    # print("--------")
    # plt.tight_layout()
    # plt.show()
    records.append({
        'id': str(dataset.ids[x]),
        'segmentation': str(rle_encode(out))
    })

000001
000002
000003
000004
000005
000006
000007
000008
000009
000010
000011
000012
000013
000014
000015
000016
000017
000018
000019
000020
000021
000022
000023
000024
000025
000026
000027
000028
000029
000030
000031
000032
000033
000034
000035
000036
000037
000038
000039
000040
000041
000042
000043
000044
000045
000046
000047
000048
000049
000050
000051
000052
000053
000054
000055
000056
000057
000058
000059
000060
000061
000062
000063
000064
000065
000066
000067
000068
000069
000070
000071
000072
000073
000074
000075
000076
000077
000078
000079
000080
000081
000082
000083
000084
000085
000086
000087
000088
000089
000090
000091
000092
000093
000094
000095
000096
000097
000098
000099
104502
106940
112782
123643
126000
126135
136374
142027
143150
144284
149617
157706
158770
169327
187649
190259
191621
191853
192744
196973
217771
222553
222596
227391
232617
238364
243578
246195
246347
250616
252446
254228
255130
255132
255544
260940
263633
268600
269087
270495
271746
274444
280079
283388

In [39]:
import pandas as pd
df = pd.DataFrame(records, columns=['id','segmentation'])
df.sort_values(by='id', inplace=True)
df.to_csv('submission.csv', index=False)
print(f"Written {len(df)} rows to submission.csv")

Written 316 rows to submission.csv


In [40]:
print(df.head(4))

       id                                       segmentation
0  000001  0 30 38 116 157 6 174 111 293 127 429 112 548 ...
1  000002  0 251 256 251 512 251 768 251 1024 251 1280 25...
2  000003                                            0 65536
3  000004                                            0 65536


In [ ]:
# ! python "/kaggle/input/run-infer/run_infernece.py" --dataset_path "/kaggle/input/cloud-masking-dataset/content/train/data" --model_path "/kaggle/working/ghaithmodel.pth"

In [17]:
class ParticipantVisibleError(Exception):
    # If you want an error message to be shown to participants, you must raise the error as a ParticipantVisibleError
    # All other errors will only be shown to the competition host. This helps prevent unintentional leakage of solution data.
    pass

def dice_coefficient(mask1: np.ndarray, mask2: np.ndarray) -> float:
    """Computes the Dice coefficient between two binary masks."""
    intersection = np.sum(mask1 * mask2)
    return (2.0 * intersection + 1e-7) / (np.sum(mask1) + np.sum(mask2) + 1e-7)  # Avoid division by zero

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """Computes the Dice score between solution and submission."""
    
    # Check if required columns exist
    required_columns = {row_id_column_name, "segmentation"}
    if not required_columns.issubset(solution.columns) or not required_columns.issubset(submission.columns):
        raise ParticipantVisibleError("Solution and submission must contain 'id' and 'segmentation' columns")
    
    # Ensure the IDs match between solution and submission
    if not solution[row_id_column_name].equals(submission[row_id_column_name]):
        raise ParticipantVisibleError("Submission IDs do not match solution IDs")
    
    # Delete the row ID column as Kaggle aligns solution and submission before passing to score()
    del solution[row_id_column_name]
    del submission[row_id_column_name]
    
    # Decode RLE masks and compute Dice score
    dice_scores = []
    for solution_seg, submission_seg in zip(solution["segmentation"], submission["segmentation"]):
        solution_mask = rle_decode(solution_seg)
        submission_mask = rle_decode(submission_seg)
        dice_scores.append(dice_coefficient(solution_mask, submission_mask))
    
    return np.mean(dice_scores)


In [42]:
sub_df = pd.read_csv('/kaggle/input/sub-csv/team_5.csv',dtype={'id': str})

In [44]:
score(sub_df,df,'id')

In [22]:
# torch.save(model, "final_model_full.pt")

In [34]:
# ! python "/kaggle/input/d/ghaithq2/final-infer/run_infernece.py" --dataset_path "/kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data" --model_path "/kaggle/working/final_model_full.pt"

In [35]:
# ! python "/kaggle/input/profiler/profiler.py" final_model_full.pt 1 4 512 512